## Task 1 — Consolidate and Name All Final `.pkl` Files

I will now simulate the saving of your pre-trained model artifacts (`final_model.pkl`, `scaler.pkl`, `label_encoder.pkl`). **Remember to replace this code with your actual fitted objects from your training process.**

In [1]:
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# --- Dummy Data Generation (Replace with your actual X_train, y_train) ---
# Simulating the PIMA Indians Diabetes Dataset features (8 features)
n_samples = 100
X = pd.DataFrame(np.random.rand(n_samples, 8) * 100, columns=[
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
])
y = pd.Series(np.random.randint(0, 2, n_samples), name='Outcome') # 0 for Not Diabetic, 1 for Diabetic

# Introduce some variability for demonstration
X['Age'] = X['Age'] * 0.5 + 20 # Age between ~20 and 70
X['Glucose'] = X['Glucose'] * 0.8 + 80 # Glucose between ~80 and 160
X['BMI'] = X['BMI'] * 0.4 + 20 # BMI between ~20 and 60

# Split data to simulate training data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Simulate fitting and saving your objects ---

# 1. StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as 'scaler.pkl'")

# 2. LabelEncoder (if you have categorical targets that were encoded)
# For binary classification like diabetes, LabelEncoder might not be strictly needed for y
# but we'll include it for completeness as per the template.
# Assuming y is already 0/1, we can create a dummy encoder for demonstration
label_encoder = LabelEncoder()
label_encoder.fit(y_train) # Fit on the target labels
joblib.dump(label_encoder, 'label_encoder.pkl')
print("Label Encoder saved as 'label_encoder.pkl'")

# 3. Final Model (RandomForestClassifier as an example)
# Apply scaling to training data before fitting the model
X_train_scaled = scaler.transform(X_train)

model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled, y_train)
joblib.dump(model, 'final_model.pkl')
print("Final Model saved as 'final_model.pkl'")

print("\nAll dummy files have been created and saved.")

Scaler saved as 'scaler.pkl'
Label Encoder saved as 'label_encoder.pkl'
Final Model saved as 'final_model.pkl'

All dummy files have been created and saved.


### Verify every file loads correctly

Now, let's load these saved files from scratch to ensure they are not corrupted and can be used independently.

In [2]:
import joblib
import numpy as np
import pandas as pd

# Load each file fresh — as if you are the Flask developer
model   = joblib.load('final_model.pkl')
scaler  = joblib.load('scaler.pkl')
encoder = joblib.load('label_encoder.pkl')

print("All .pkl files loaded successfully.")

# Quick sanity check — make one prediction to confirm the model loaded correctly
# Replace these with valid values for your project, matching your feature order
# Example values for a diabetes model:
# Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
test_input_raw = np.array([[2, 148, 72, 35, 0, 33.6, 0.627, 50]])

# It's crucial to scale the input data before prediction, just like during training.
# Convert to DataFrame to maintain column names if your scaler expects it, otherwise a numpy array is fine.
test_input_df = pd.DataFrame(test_input_raw, columns=[
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
])
scaled_test_input = scaler.transform(test_input_df)

sample_prediction = model.predict(scaled_test_input)[0]
sample_probabilities = model.predict_proba(scaled_test_input)[0]

# Decode prediction if your model output is encoded
predicted_label = encoder.inverse_transform([sample_prediction])[0]

print("Load test passed. Sample prediction:")
print(f"  Predicted Class: {predicted_label}")
print(f"  Confidence: {np.max(sample_probabilities)*100:.1f}%")

All .pkl files loaded successfully.
Load test passed. Sample prediction:
  Predicted Class: 0
  Confidence: 62.0%


### Write a handoff checklist in a markdown cell

This checklist tells the M3 team exactly what to load and in what order.

## Files required for prediction

| File | Purpose |
|---:|:---|
| `final_model.pkl` | Trained Random Forest classifier |
| `scaler.pkl` | StandardScaler fitted on training features |
| `label_encoder.pkl` | LabelEncoder fitted on target variable `Outcome` |

## Input columns (in this exact order)
`Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age`

## Output
`{ "prediction": "Not Diabetic" or "Diabetic", "confidence": "84.7%" }`

## Task 2 — Write and Test a Single `predict(inputs)` Function

This function will serve as the primary interface for your model, taking raw input and returning a prediction. I've included the template provided in the instructions, which you can customize further if needed. Note the addition of a `try-except` block to gracefully handle invalid inputs, which is crucial for Task 3.

In [7]:
import joblib
import pandas as pd
import numpy as np

def predict(inputs: dict) -> dict:
    """
    Takes a raw input dictionary (exactly what the Flask form will send)
    and returns a dictionary with the prediction and confidence.

    Parameters
    ----------
    inputs : dict
        Raw values from the user, e.g.
        {'Glucose': 148, 'BMI': 33.6, 'Age': 50, ...}

    Returns
    -------
    dict
        e.g. {'prediction': 'Diabetic', 'confidence': '84.7%'}
    """
    try:
        # Step 1 — Load all required .pkl files
        # Load these inside the function to ensure it's self-contained
        model  = joblib.load('final_model.pkl')
        scaler = joblib.load('scaler.pkl')
        encoder = joblib.load('label_encoder.pkl')

        # Define class labels based on your LabelEncoder's classes
        # Assuming binary classification for simplicity (0: Not Diabetic, 1: Diabetic)
        # You can get this directly from encoder.classes_ if you fit on string labels
        class_labels = {0: 'Not Diabetic', 1: 'Diabetic'} # Adjust based on your actual labels and their mapping

        # Step 2 — Convert the input dict into a DataFrame
        # This preserves column names so the model knows which value is which
        input_df = pd.DataFrame([inputs])

        # Step 3 — Apply preprocessing in the EXACT same order as training
        # Ensure the column order is consistent with how the scaler was fitted
        feature_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
        input_df_ordered = input_df[feature_columns] # Ensure correct order
        input_df_scaled = scaler.transform(input_df_ordered)

        # Step 4 — Make the prediction
        prediction_idx   = model.predict(input_df_scaled)[0]
        probabilities = model.predict_proba(input_df_scaled)[0]

        # Step 5 — Return a clean, human-readable dict
        predicted_label = encoder.inverse_transform([prediction_idx])[0]

        return {
            'prediction': class_labels[predicted_label],
            'confidence': f"{max(probabilities) * 100:.1f}%"
        }
    except Exception as e:
        # Catch any errors during prediction and return a structured error message
        return {'prediction': 'Error', 'confidence': 'N/A', 'error': str(e)}

print("The 'predict' function has been defined.")

The 'predict' function has been defined.


### Test the function with one quick call

Let's ensure the `predict()` function works as expected with a single sample input.

In [4]:
sample_input = {
    'Pregnancies': 2,
    'Glucose': 148,
    'BloodPressure': 72,
    'SkinThickness': 35,
    'Insulin': 0,
    'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age': 50
}

result = predict(sample_input)
print(result)
# Expected output for this example input might vary depending on your specific model training,
# but it should return a dictionary with 'prediction' and 'confidence'.

{'prediction': 'Not Diabetic', 'confidence': '62.0%'}


## Task 3 — Run 10 Test Cases Through `predict()` and Log Results

We will now define 10 test cases to rigorously evaluate the `predict()` function's behavior across various inputs, including an invalid input to ensure error handling. The results will be logged in a table.

In [5]:
import pandas as pd

# Define your 10 test cases as a list of dicts
test_cases = [
    # Case 1 — Normal / low risk
    {'Pregnancies': 1, 'Glucose': 89,  'BloodPressure': 66,
     'SkinThickness': 23, 'Insulin': 94,  'BMI': 28.1,
     'DiabetesPedigreeFunction': 0.167, 'Age': 21},

    # Case 2 — Normal
    {'Pregnancies': 0, 'Glucose': 95,  'BloodPressure': 60,
     'SkinThickness': 18, 'Insulin': 58,  'BMI': 22.5,
     'DiabetesPedigreeFunction': 0.112, 'Age': 24},

    # Case 3 — Normal / average
    {'Pregnancies': 3, 'Glucose': 110, 'BloodPressure': 70,
     'SkinThickness': 25, 'Insulin': 80,  'BMI': 27.0,
     'DiabetesPedigreeFunction': 0.300, 'Age': 30},

    # Case 4 — High risk
    {'Pregnancies': 8, 'Glucose': 183, 'BloodPressure': 64,
     'SkinThickness': 0,  'Insulin': 0,   'BMI': 23.3,
     'DiabetesPedigreeFunction': 0.672, 'Age': 32},

    # Case 5 — High risk
    {'Pregnancies': 10,'Glucose': 199, 'BloodPressure': 90,
     'SkinThickness': 50, 'Insulin': 200, 'BMI': 45.0,
     'DiabetesPedigreeFunction': 2.1,   'Age': 63},

    # Case 6 — Edge: minimum values
    {'Pregnancies': 0, 'Glucose': 44,  'BloodPressure': 24,
     'SkinThickness': 7,  'Insulin': 14,  'BMI': 18.2,
     'DiabetesPedigreeFunction': 0.078, 'Age': 21},

    # Case 7 — Edge: maximum values
    {'Pregnancies': 17,'Glucose': 199, 'BloodPressure': 122,
     'SkinThickness': 99, 'Insulin': 846, 'BMI': 67.1,
     'DiabetesPedigreeFunction': 2.42,  'Age': 81},

    # Case 8 — Known test row (y_test = 0, not diabetic)
    # Replace these with actual values from your X_test where y_test = 0
    {'Pregnancies': 1, 'Glucose': 103, 'BloodPressure': 30,
     'SkinThickness': 38, 'Insulin': 83,  'BMI': 43.3,
     'DiabetesPedigreeFunction': 0.183, 'Age': 33},

    # Case 9 — Known test row (y_test = 1, diabetic)
    # Replace these with actual values from your X_test where y_test = 1
    {'Pregnancies': 2, 'Glucose': 155, 'BloodPressure': 52,
     'SkinThickness': 27, 'Insulin': 540, 'BMI': 38.7,
     'DiabetesPedigreeFunction': 0.240, 'Age': 25},

    # Case 10 — Invalid input (should NOT crash — must return error message)
    # For this example, let's make Glucose a string, which should cause a ValueError
    {'Pregnancies': 1, 'Glucose': 'invalid', 'BloodPressure': 70,
     'SkinThickness': 25, 'Insulin': 80,  'BMI': 27.0,
     'DiabetesPedigreeFunction': 0.3,   'Age': 30},
]

# Run all 10 and log results
results = []
for i, case in enumerate(test_cases, 1):
    result = predict(case) # predict function already has try/except
    results.append({
        'Case': i,
        'Prediction': result.get('prediction', 'Error'),
        'Confidence': result.get('confidence', 'N/A'),
        'Error_Message': result.get('error', '') # Log the error message if any
    })

log_df = pd.DataFrame(results)

# Determine Status based on whether an error occurred
log_df['Status'] = log_df['Prediction'].apply(lambda x: 'ERROR' if x == 'Error' else 'PASS')

print(log_df.to_string(index=False))

 Case   Prediction Confidence                                Error_Message Status
    1 Not Diabetic      50.0%                                                PASS
    2 Not Diabetic      58.0%                                                PASS
    3 Not Diabetic      61.0%                                                PASS
    4 Not Diabetic      64.0%                                                PASS
    5 Not Diabetic      76.0%                                                PASS
    6     Diabetic      52.0%                                                PASS
    7 Not Diabetic      66.0%                                                PASS
    8 Not Diabetic      65.0%                                                PASS
    9 Not Diabetic      59.0%                                                PASS
   10        Error        N/A could not convert string to float: 'invalid'  ERROR


## Task 4 — Write the Model Summary Card

This card documents everything about your final model in plain English for the Flask integration team. Please fill in every section with your project's real values.

```
## Model Summary Card

### Project
<!-- Your project name and domain -->
Diabetes Risk Predictor · Healthcare

### Algorithm
Random Forest Classifier (tuned with GridSearchCV)

### Dataset
PIMA Indians Diabetes Dataset · 768 rows · 8 features

### Final Performance
| Metric | Score |
|---|---|
| Accuracy | 79.2% |
| F1-Score (weighted) | 0.787 |
| Cross-validation mean | 0.774 ± 0.031 |

### Input Features (in this exact order)
| Column | Type | Example value |
|---|---|---|
| Pregnancies | int | 2 |
| Glucose | float | 148.0 |
| BloodPressure | float | 72.0 |
| SkinThickness | float | 35.0 |
| Insulin | float | 0.0 |
| BMI | float | 33.6 |
| DiabetesPedigreeFunction | float | 0.627 |
| Age | int | 50 |

### Required .pkl Files
| File | Contents |
|---|---|
| final_model.pkl | Trained Random Forest (n_estimators=200, max_depth=10) |
| scaler.pkl | StandardScaler fitted on X_train |
| label_encoder.pkl | LabelEncoder fitted on target variable `Outcome` |

### Sample Input
```python
{
    'Pregnancies': 2,
    'Glucose': 148,
    'BloodPressure': 72,
    'SkinThickness': 35,
    'Insulin': 0,
    'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age': 50
}
```

### Sample Output
```python
{
    'prediction': 'Diabetic',
    'confidence': '84.7%',
    'top_features': ['Glucose', 'BMI', 'Age']
}
```

### How to use
```python
import joblib
result = predict(your_input_dict)
```
```

Remember to replace the placeholder values with the actual details from your project, especially the performance metrics, model parameters, and example input/output if they differ from the diabetes predictor example.

## Task 5 — Push All Files to GitHub and Submit InnoTrack Report

This task requires actions outside of this notebook environment. Please follow these steps to finalize your project handoff:

### Folder structure to push
Your GitHub repository must have this structure:

```
your-repo/
│
├── M2_model_development.ipynb   ← your full Day 6-10 notebook
│
├── models/
│   ├── final_model.pkl
│   ├── scaler.pkl               ← if used
│   ├── label_encoder.pkl        ← if used
│   └── tfidf_vectorizer.pkl     ← if used (Shaik only)
│
├── charts/
│   ├── feature_importance.png
│   ├── confusion_matrix.png     ← or residual_plot.png for regressors
│   └── validation_curve.png
│
└── comparison.csv               ← your Day 7 model comparison table
```

### Git commands to push everything
```bash
# Navigate to your project folder
cd your-project-folder

# Stage all new and changed files
git add .

# Commit with a clear message
git commit -m "M2 complete: final model, predict function, test cases, summary card"

# Push to GitHub
git push origin main
```

### Verify your repo on GitHub
After pushing, open your GitHub repo in a browser and confirm:

- `M2_model_development.ipynb` is present and visible
- `models/` folder exists and contains all `.pkl` files
- `charts/` folder contains all PNG charts
- `comparison.csv` is present

If any file is missing, add it and push again before submitting.

### Submit on InnoTrack
Your InnoTrack submission must include:

1. Your GitHub repository link
2. A screenshot of your final model's metric score (from the notebook output)
3. A screenshot of your comparison table (from `comparison.csv` or the notebook)

---

### Re-verifying all `.pkl` files load correctly

As requested, here's an independent check to ensure all model artifact files can be loaded without issues. This simulates how a deployment environment would load your model.

In [6]:
import joblib
import numpy as np
import pandas as pd

# Load each file fresh to ensure they are accessible and not corrupted
# Note: If you didn't use a scaler or encoder, you might comment out or remove those lines.
model   = joblib.load('final_model.pkl')
scaler  = joblib.load('scaler.pkl')
encoder = joblib.load('label_encoder.pkl')

print("All .pkl files loaded successfully in this fresh check.")

# Quick sanity check with a sample prediction, just like in your previous verification
test_input_raw = np.array([[2, 148, 72, 35, 0, 33.6, 0.627, 50]])

test_input_df = pd.DataFrame(test_input_raw, columns=[
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
])
scaled_test_input = scaler.transform(test_input_df)

sample_prediction = model.predict(scaled_test_input)[0]
sample_probabilities = model.predict_proba(scaled_test_input)[0]

predicted_label = encoder.inverse_transform([sample_prediction])[0]

print("Sample prediction from freshly loaded model:")
print(f"  Predicted Class: {predicted_label}")
print(f"  Confidence: {np.max(sample_probabilities)*100:.1f}%")
print("Verification complete.")

All .pkl files loaded successfully in this fresh check.
Sample prediction from freshly loaded model:
  Predicted Class: 0
  Confidence: 62.0%
Verification complete.
